# PPE Detector — YOLOv8 Training on Colab GPU
This notebook trains YOLOv8n, YOLOv8s, YOLOv8m on the PPE dataset.
**Step 1:** Runtime → Change runtime type → T4 GPU
**Step 2:** Mount Drive to persist weights
**Step 3:** Run all cells

In [ ]:
!nvidia-smi
!pip install ultralytics==8.3.0 -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ppe_detector/weights', exist_ok=True)

In [ ]:
# Expects /content/drive/MyDrive/ppe_detector/dataset.zip
# (zip of data/processed/ with images/train,val,test + labels/train,val,test)
!unzip -q /content/drive/MyDrive/ppe_detector/dataset.zip -d /content/dataset
!ls /content/dataset

In [ ]:
yaml_content = '''
path: /content/dataset
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: helmet
  2: vest
  3: no-helmet
  4: no-vest
  5: boots
'''
open('/content/dataset.yaml','w').write(yaml_content)
print(open('/content/dataset.yaml').read())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(data='/content/dataset.yaml', epochs=60, imgsz=640, batch=32,
            device=0, project='/content/runs', name='yolov8n_ppe',
            plots=True, save=True, exist_ok=True)
import shutil
shutil.copytree('/content/runs/yolov8n_ppe',
                '/content/drive/MyDrive/ppe_detector/weights/yolov8n_ppe',
                dirs_exist_ok=True)

In [ ]:
model = YOLO('yolov8s.pt')
model.train(data='/content/dataset.yaml', epochs=60, imgsz=640, batch=32,
            device=0, project='/content/runs', name='yolov8s_ppe',
            plots=True, save=True, exist_ok=True)
shutil.copytree('/content/runs/yolov8s_ppe',
                '/content/drive/MyDrive/ppe_detector/weights/yolov8s_ppe',
                dirs_exist_ok=True)

In [ ]:
model = YOLO('yolov8m.pt')
model.train(data='/content/dataset.yaml', epochs=60, imgsz=640, batch=16,
            device=0, project='/content/runs', name='yolov8m_ppe',
            plots=True, save=True, exist_ok=True)
shutil.copytree('/content/runs/yolov8m_ppe',
                '/content/drive/MyDrive/ppe_detector/weights/yolov8m_ppe',
                dirs_exist_ok=True)

In [ ]:
import json
results = {}
for name in ['yolov8n_ppe','yolov8s_ppe','yolov8m_ppe']:
    m = YOLO(f'/content/runs/{name}/weights/best.pt')
    r = m.val(data='/content/dataset.yaml', split='test')
    results[name] = {
        'mAP50': float(r.box.map50),
        'mAP50_95': float(r.box.map),
        'precision': float(r.box.mp),
        'recall': float(r.box.mr),
    }
json.dump(results, open('/content/drive/MyDrive/ppe_detector/metrics.json','w'), indent=2)
print(json.dumps(results, indent=2))

In [ ]:
!mkdir -p /content/drive/MyDrive/ppe_detector/plots
!cp /content/runs/yolov8n_ppe/*.png /content/drive/MyDrive/ppe_detector/plots/n_*.png
!cp /content/runs/yolov8s_ppe/*.png /content/drive/MyDrive/ppe_detector/plots/s_*.png
!cp /content/runs/yolov8m_ppe/*.png /content/drive/MyDrive/ppe_detector/plots/m_*.png
# These PNGs are results.png, confusion_matrix.png, PR_curve.png, F1_curve.png, val_batch*_pred.jpg

After running all cells, download the entire `/content/drive/MyDrive/ppe_detector/` folder and place:
- `weights/*` → `models/checkpoints/`
- `plots/*` → `reports/figures/colab_plots/`
- `metrics.json` → `reports/colab_metrics.json`